# Lab 03 — Part 3: Polymer in Poiseuille Flow (standalone)

This notebook is **self-contained**: it installs ESPResSo from source, runs a
quick LBM sanity check, then simulates a polymer in Poiseuille flow and
investigates two key results:

1. **Free diffusion vs Rouse theory** — does the LBM diffusion coefficient
   match the Rouse prediction $D = k_BT/(N\gamma)$, and why does it differ?

2. **Flow velocity profile** — verify that the simulated Poiseuille profile
   and $v_{max}$ match the analytical prediction $v_{max} = fH^2/8\eta$.

**Runtime**: ~30–40 min on Colab CPU (3 force densities).


## 0. Installation

In [ ]:
# Install system and Python dependencies
!apt-get update -qq
!apt-get install -y cmake g++ libfftw3-dev libhdf5-dev libboost-all-dev \
    openmpi-bin libopenmpi-dev
!pip install -q numpy scipy matplotlib h5py


In [ ]:
# Clone ESPResSo 4.2
!git clone --recursive --single-branch -b 4.2 \
    https://github.com/espressomd/espresso.git /content/espresso


In [ ]:
# Build ESPResSo (takes ~15 min on Colab CPU)
%%bash
mkdir -p /content/espresso/build
cd /content/espresso/build
cmake .. -DPYTHON=ON -DENABLE_PYTHON=ON \
         -DCMAKE_INSTALL_PREFIX=/content/espresso/install \
         -DWITH_CUDA=OFF
make -j2
make install


In [ ]:
# Add ESPResSo to Python path (dynamic — works regardless of Python version)
import sys, glob
matches = glob.glob("/content/espresso/install/**/espressomd", recursive=True)
if matches:
    sys.path.insert(0, matches[0].rsplit("/espressomd", 1)[0])
else:
    raise RuntimeError("espressomd not found — did the build finish without errors?")

import espressomd
espressomd.assert_features([])  # lightweight check that the module loaded correctly
print("ESPResSo imported successfully.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import espressomd
import espressomd.lb
import espressomd.lbboundaries
import espressomd.shapes
import espressomd.constraints
import espressomd.polymer
import espressomd.interactions
import espressomd.observables
import espressomd.accumulators
print("All imports OK")


## 1. LBM sanity check — Poiseuille profile (no polymer)

Before adding the polymer we verify that the LBM fluid with bounce-back walls
produces the correct parabolic velocity profile.  
`kT=0` removes thermal noise so the profile is clean.

**Runs in ~1–2 min.**


In [ ]:
# ── Parameters ───────────────────────────────────────────────────────────────
BOX   = 16       # box size (same in all directions)
AGRID = 1.0      # LB grid spacing
VISC  = 5.0      # kinematic viscosity
DENS  = 1.0
DT    = 0.01
F_TEST = 0.003   # body-force density for the sanity check

# Analytical v_max = F * H^2 / (8 eta),  H = BOX - 2*AGRID (channel width)
eta   = VISC * DENS
H     = BOX - 2 * AGRID
v_max_ana = F_TEST * H**2 / (8 * eta)
print(f"Analytical v_max = {v_max_ana:.5f}")


In [ ]:
# ── Build system + LBM fluid with bounce-back walls ──────────────────────────
system_check = espressomd.System(box_l=[BOX]*3)
system_check.periodicity     = [True, True, True]
system_check.time_step       = DT
system_check.cell_system.skin = 0.4

# Force in Y, walls in X (same convention as Part 2 of the main lab)
lbf_check = espressomd.lb.LBFluid(
    kT=0, agrid=AGRID, dens=DENS, visc=VISC, tau=DT,
    ext_force_density=[0.0, F_TEST, 0.0])
system_check.actors.add(lbf_check)

system_check.lbboundaries.add(espressomd.lbboundaries.LBBoundary(
    shape=espressomd.shapes.Wall(normal=[1, 0, 0], dist=AGRID)))
system_check.lbboundaries.add(espressomd.lbboundaries.LBBoundary(
    shape=espressomd.shapes.Wall(normal=[-1, 0, 0], dist=-(BOX - AGRID))))

system_check.integrator.run(5000)
print("Integration done.")


In [ ]:
# ── Read velocity profile and plot ───────────────────────────────────────────
x_cen  = np.arange(BOX) + 0.5
v_sim  = np.array([lbf_check[j, 0, 0].velocity[1] for j in range(BOX)])

x0, x1 = AGRID, BOX - AGRID
v_ana  = np.where((x_cen > x0) & (x_cen < x1),
                  F_TEST / (2*eta) * (x_cen - x0) * (x1 - x_cen), 0.0)

plt.figure(figsize=(5, 4))
plt.plot(v_sim, x_cen, 'o', ms=5, label='LBM')
plt.plot(v_ana, x_cen, '--',      label='Analytical')
plt.xlabel(r"$v_y$  (flow direction)")
plt.ylabel(r"$x$  (wall-normal)")
plt.title("Poiseuille profile — LBM sanity check")
plt.legend(); plt.tight_layout(); plt.show()

print(f"v_max  simulation : {v_sim.max():.5f}")
print(f"v_max  analytical : {v_max_ana:.5f}")
print(f"relative error    : {abs(v_sim.max()-v_max_ana)/v_max_ana*100:.2f}%")


## 2. Polymer in Poiseuille flow

We add a short polymer chain (Rouse, 10 beads) to the Poiseuille flow and
measure two things:

1. **Free diffusion D₀** — compare to the Rouse model prediction
   $D_{Rouse} = k_BT / (N \cdot \gamma)$

2. **Polymer drift velocity** — verify that the polymer is carried at the
   spatial mean of the parabolic flow profile:
   $v_{drift} = \frac{2}{3} v_{max}$


In [ ]:
# ── Simulation parameters ─────────────────────────────────────────────────────
N_MON    = 10       # monomers
BOX_L    = 16.0     # cubic box
LB_AGRID = 1.0
LB_VISC  = 5.0
LB_DENS  = 1.0
GAMMA    = 5.0
KT       = 1.0
TIME_STEP = 0.01

LOOPS    = 400      # production loops
STEPS    = 200      # integrator steps per loop  → 80 k steps total
TAU_MAX  = LOOPS * STEPS * TIME_STEP * 0.5

FORCE_DENSITIES = [0.0, 0.003, 0.010]  # Pe ≈ 0, moderate, high

# Derived channel geometry
ETA  = LB_VISC * LB_DENS
X0   = LB_AGRID
X1   = BOX_L - LB_AGRID
H_CH = X1 - X0      # effective channel width

print(f"Channel width H = {H_CH:.1f}")
print(f"tau_max = {TAU_MAX:.1f} time units")


In [ ]:
# ── Simulation function ───────────────────────────────────────────────────────
# ESPResSo only allows ONE System per Python session.  We reuse the one
# already created in Section 1 (system_check) rather than creating a new one.

_fene = None

def run_poiseuille(f_drive, n_monomers=N_MON, seed=42):
    global _fene

    # ── Reuse existing System, reset state ───────────────────────────────
    system = system_check
    system.part.clear()
    system.thermostat.turn_off()
    system.auto_update_accumulators.clear()
    system.constraints.clear()
    system.lbboundaries.clear()
    system.actors.clear()

    # ── Interactions ──────────────────────────────────────────────────────
    system.non_bonded_inter[0, 0].lennard_jones.set_params(
        epsilon=1.0, sigma=1.0, cutoff=2.0**(1/6), shift="auto")

    if _fene is None:
        _fene = espressomd.interactions.FeneBond(k=7, r_0=1, d_r_max=2)
        system.bonded_inter.add(_fene)

    # ── Polymer ───────────────────────────────────────────────────────────
    start_pos = np.array([[BOX_L/2, BOX_L/2, BOX_L/2]])
    positions = espressomd.polymer.linear_polymer_positions(
        n_polymers=1, beads_per_chain=n_monomers,
        bond_length=1.0, seed=seed, min_distance=0.9,
        start_positions=start_pos)

    pids = []
    for i, pos in enumerate(positions[0]):
        p = system.part.add(pos=pos, type=0)
        if i > 0:
            p.add_bond((_fene, pids[-1]))
        pids.append(p.id)

    # ── LBM fluid: force in Y, walls in X ────────────────────────────────
    lbf = espressomd.lb.LBFluid(
        kT=KT, seed=seed, agrid=LB_AGRID,
        dens=LB_DENS, visc=LB_VISC, tau=TIME_STEP,
        ext_force_density=[0.0, f_drive, 0.0])
    system.actors.add(lbf)

    system.lbboundaries.add(espressomd.lbboundaries.LBBoundary(
        shape=espressomd.shapes.Wall(normal=[1, 0, 0], dist=LB_AGRID)))
    system.lbboundaries.add(espressomd.lbboundaries.LBBoundary(
        shape=espressomd.shapes.Wall(normal=[-1, 0, 0],
                                     dist=-(BOX_L - LB_AGRID))))

    system.constraints.add(espressomd.constraints.ShapeBasedConstraint(
        shape=espressomd.shapes.Wall(normal=[1, 0, 0], dist=1.5),
        particle_type=0, penetrable=False))
    system.constraints.add(espressomd.constraints.ShapeBasedConstraint(
        shape=espressomd.shapes.Wall(normal=[-1, 0, 0],
                                     dist=-(BOX_L - 1.5)),
        particle_type=0, penetrable=False))

    # ── Equilibration ─────────────────────────────────────────────────────
    system.integrator.set_steepest_descent(
        f_max=0, gamma=0.1, max_displacement=0.05)
    system.integrator.run(500)

    system.thermostat.set_lb(LB_fluid=lbf, gamma=GAMMA, seed=seed)
    system.integrator.set_vv()
    system.integrator.run(5000)

    # ── MSD correlator ────────────────────────────────────────────────────
    com_pos = espressomd.observables.ComPosition(ids=pids)
    msd_cor = espressomd.accumulators.Correlator(
        obs1=com_pos, tau_lin=16, tau_max=TAU_MAX, delta_N=5,
        corr_operation="square_distance_componentwise",
        compress1="discard1")
    system.auto_update_accumulators.add(msd_cor)

    # ── Production — time-average the velocity profile ────────────────────
    # A single LBM snapshot at kT=1 is dominated by thermal noise (v_thermal~1
    # vs v_flow~0.015).  Averaging over all production steps recovers the mean.
    v_accum = np.zeros(int(BOX_L))
    n_accum = 0
    for step in range(LOOPS):
        system.integrator.run(STEPS)
        if step % 5 == 0:   # sample every 5 loops
            v_accum += np.array([lbf[j, 0, 0].velocity[1]
                                  for j in range(int(BOX_L))])
            n_accum += 1
        if step % 100 == 0:
            print(f'  loop {step}/{LOOPS}', flush=True)

    msd_cor.finalize()
    msd  = np.array(msd_cor.result())
    lag  = np.array(msd_cor.lag_times())
    v_prof = v_accum / n_accum   # time-averaged profile

    return msd, lag, v_prof

print('Function defined.')


In [ ]:
# ── Verify walls survive the polymer thermostat ───────────────────────────────
# At kT=1 the thermal velocity (~1.0) swamps the flow signal (~0.015) even
# after time-averaging.  We use a large force (f=1.0, v_max~2.5) so the
# parabola is clearly visible above the noise.
_, _, v_chk = run_poiseuille(f_drive=1.0)

x_cen  = np.arange(int(BOX_L)) + 0.5
f_big  = 1.0
v_ana  = np.where((x_cen > X0) & (x_cen < X1),
                  f_big / (2*ETA) * (x_cen - X0) * (X1 - x_cen), 0.0)

plt.figure(figsize=(5, 4))
plt.plot(v_chk, x_cen, 'o', ms=5, label='LBM + polymer thermostat')
plt.plot(v_ana, x_cen, '--',      label='Analytical')
plt.xlabel(r'$v_y$  (flow direction)')
plt.ylabel(r'$x$  (wall-normal)')
plt.title('Wall check at large force (f=1.0, high SNR)')
plt.legend(); plt.tight_layout(); plt.show()

v_max_ana = f_big * (X1 - X0)**2 / (8*ETA)
print(f'v_max  sim={v_chk.max():.4f}  analytical={v_max_ana:.4f}')
print('Walls OK if sim ~ analytical')


In [ ]:
# ── Pe scan ───────────────────────────────────────────────────────────────────
results = {}
for f_drive in FORCE_DENSITIES:
    v_max = f_drive * H_CH**2 / (8*ETA)
    print(f"\nf_drive={f_drive}  v_max={v_max:.5f}")
    msd, lag, _ = run_poiseuille(f_drive=f_drive)
    results[f_drive] = {"msd": msd, "lag": lag, "v_max": v_max}

print("\nDone.")


In [ ]:
# ── Free diffusion D0 vs Rouse theory ────────────────────────────────────────
KT        = 1.0
N_MON     = 10
GAMMA     = 5.0
LB_AGRID  = 1.0
LB_VISC   = 5.0
LB_DENS   = 1.0
ETA       = LB_VISC * LB_DENS
X0        = LB_AGRID
X1        = 16.0 - LB_AGRID
H_CH      = X1 - X0
TIME_STEP = 0.01
BOX_L     = 16.0

def fit_D(lag, msd_1d, frac_lo=0.35, frac_hi=0.65):
    n = len(lag)
    lo, hi = int(n * frac_lo), int(n * frac_hi)
    if hi <= lo + 2:
        return np.nan
    slope, _ = np.polyfit(lag[lo:hi], msd_1d[lo:hi], 1)
    return slope / 2.0

D_rouse = KT / (N_MON * GAMMA)
lag0    = results[0.0]['lag']
msd0    = results[0.0]['msd']
D0_sim  = fit_D(lag0, msd0[:, 1])

print(f'Rouse prediction  D_Rouse = {D_rouse:.4f}')
print(f'LBM simulation    D0      = {D0_sim:.4f}')
print(f'Ratio D0_sim / D_Rouse   = {D0_sim/D_rouse:.3f}')
print()

# ── v_max simulation vs analytical ────────────────────────────────────────────
f_list, v_max_sim_list, v_max_ana_list = [], [], []

for f_drive in [0.003, 0.010]:
    system_check.thermostat.turn_off()
    system_check.auto_update_accumulators.clear()
    system_check.constraints.clear()
    system_check.part.clear()
    system_check.actors.clear()
    system_check.lbboundaries.clear()

    lbf_test = espressomd.lb.LBFluid(
        kT=0, agrid=LB_AGRID, dens=LB_DENS, visc=LB_VISC, tau=TIME_STEP,
        ext_force_density=[0.0, f_drive, 0.0])
    system_check.actors.add(lbf_test)
    system_check.lbboundaries.add(espressomd.lbboundaries.LBBoundary(
        shape=espressomd.shapes.Wall(normal=[1, 0, 0], dist=LB_AGRID)))
    system_check.lbboundaries.add(espressomd.lbboundaries.LBBoundary(
        shape=espressomd.shapes.Wall(normal=[-1, 0, 0], dist=-(BOX_L - LB_AGRID))))
    system_check.integrator.set_vv()
    system_check.integrator.run(5000)

    v_prof    = np.array([lbf_test[j, 0, 0].velocity[1] for j in range(int(BOX_L))])
    v_max_sim = v_prof.max()
    v_max_ana = f_drive * H_CH**2 / (8 * ETA)
    f_list.append(f_drive)
    v_max_sim_list.append(v_max_sim)
    v_max_ana_list.append(v_max_ana)
    print(f'f={f_drive:.4f}  v_max_sim={v_max_sim:.5f}  '
          f'v_max_ana={v_max_ana:.5f}  ratio={v_max_sim/v_max_ana:.4f}')


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Left: Rouse vs simulation
labels = ['Rouse\nprediction', 'LBM\nsimulation']
values = [D_rouse, D0_sim]
bars   = ax1.bar(labels, values, color=['steelblue', 'orange'], width=0.4)
for bar, v in zip(bars, values):
    ax1.text(bar.get_x() + bar.get_width()/2, v + 0.0005,
             f'{v:.4f}', ha='center', va='bottom')
ax1.set_ylabel(r'$D_0$')
ax1.set_title('Free diffusion: Rouse vs LBM simulation')
ax1.set_ylim(0, max(values) * 1.25)

# Right: v_max simulation vs analytical
x, w = np.arange(len(f_list)), 0.35
ax2.bar(x - w/2, v_max_ana_list, w, label=r'Analytical $fH^2/8\eta$', color='steelblue')
ax2.bar(x + w/2, v_max_sim_list, w, label='LBM simulation',           color='orange')
ax2.set_xticks(x)
ax2.set_xticklabels([f'f={f:.3f}' for f in f_list])
ax2.set_ylabel(r'$v_{max}$')
ax2.set_title(r'$v_{max}$: simulation vs analytical')
ax2.legend()

plt.tight_layout()
plt.savefig('diffusion_vmax_comparison.png', dpi=120)
plt.show()


## 3. Diffusion anisotropy in flow


In [ ]:
f_vals, D_par_vals, D_perp_vals = [], [], []

for f_drive, res in results.items():
    lag   = res['lag']
    msd   = res['msd']
    v_max = res['v_max']
    v_mean     = (2/3) * v_max
    msd_y_corr = msd[:, 1] - v_mean**2 * lag**2
    D_par  = fit_D(lag, msd_y_corr)
    D_x    = fit_D(lag, msd[:, 0])
    D_z    = fit_D(lag, msd[:, 2])
    D_perp = (D_x + D_z) / 2
    f_vals.append(f_drive)
    D_par_vals.append(D_par)
    D_perp_vals.append(D_perp)
    print(f'f={f_drive:.4f}  D_∥={D_par:.4f}  D_⊥={D_perp:.4f}  '
          f'D_∥/D_⊥={D_par/D_perp:.2f}')

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(f_vals, D_par_vals,  'o-', label=r'$D_\parallel$ (flow direction)')
ax.plot(f_vals, D_perp_vals, 's-', label=r'$D_\perp$ (transverse mean)')
ax.axhline(D0_sim, color='gray', linestyle='--', label=r'$D_0$ (free, f=0)')
ax.set_xlabel('Force density f')
ax.set_ylabel('Diffusion coefficient')
ax.set_title('Diffusion anisotropy in Poiseuille flow')
ax.legend()
plt.tight_layout()
plt.savefig('diffusion_anisotropy.png', dpi=120)
plt.show()


## Questions

**Question 1 — Hydrodynamic interactions**

The Rouse model predicts $D = k_BT/(N\gamma)$, but the LBM simulation gives a higher value. Explain why. What physical interactions does the LBM fluid include that the Rouse model ignores? How does this relate to the difference between the Rouse and Zimm models discussed in Part 1?

---

**Question 2 — Diffusion anisotropy**

At $f=0$ the polymer diffuses isotropically. At $f>0$, $D_\parallel$ increases while $D_\perp$ stays below $D_0$. Explain the physical mechanism behind each observation. Why is $D_\perp$ already suppressed below $D_0$ even at $f=0$?
